# EGU26 Hydrological GNN — Ablation & Baseline Notebook (v2)
**목적**: 미호천 유역 Chl-a 예측 시계열 모델 아키텍처 비교 (Transformer vs GRU vs LightGBM)
**주요 반영 사항**:
- Log 변환(np.log1p) 및 Expm1 역변환을 통한 R², MAE 스케일 정규화 완료
- Lookback Window 짧게 재설정
**실행 환경**: Google Colab (GPU 런타임 권장)

---
## 실행 전 체크리스트
1. `런타임 → 런타임 유형 변경 → GPU` 설정
2. 깃허브 최신 버전 유지


## 1. 환경 세팅

In [ ]:
# 깃허브에서 최신 코드 다운로드 및 작업 폴더 이동
!git clone https://github.com/kona0107/EGU26-SWAT-GNN.git
%cd EGU26-SWAT-GNN

# 향후 로컬에서 수정 후 푸시했다면 아래 명령어 주석을 해제하고 실행하세요.
# !git pull origin main

In [ ]:
# 의존 패키지 설치 (PyTorch Geometric)
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')

!pip install torch_geometric -q
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html -q

!pip install lightgbm -q


In [ ]:
import sys
import os

# 경로 설정 (git clone 한 위치 기준)
PROJECT_ROOT = '.'
MODULE_PATH  = os.path.join(PROJECT_ROOT, 'script/src/gnn_project')
DATA_PATH    = os.path.join(PROJECT_ROOT, 'script/data')

OUTLET_CSV   = os.path.join(DATA_PATH, 'outlet_lag_pcp.csv')
UPSTREAM_CSV = os.path.join(DATA_PATH, 'upstream_lag_pcp.csv')

sys.path.insert(0, MODULE_PATH)
print('Module path:', MODULE_PATH)
print('Outlet  CSV:', os.path.exists(OUTLET_CSV))
print('Upstream CSV:', os.path.exists(UPSTREAM_CSV))


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from data.dataset import (
    load_real_data, prepare_and_split_data,
    FEATURE_DIM, OUTLET_IDX, N_NODES
)
from models.temporal import TransformerBaseline
from models.st_gcn   import SpatioTemporalHybridGNN

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## 2. 데이터 로딩 및 분할

In [ ]:
# ── 하이퍼파라미터 ───────────────────────────
LOOKBACK    = 5   # 관측 윈도우 크기 축소 (주 단위 최적 4~6 탐색)
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.15
BATCH_SIZE  = 16
# ─────────────────────────────────────────────

raw_features, dates = load_real_data(OUTLET_CSV, UPSTREAM_CSV)
print(f'총 관측 날짜 수 (T) : {len(dates)}')
print(f'raw_features shape : {raw_features.shape}  → 기대: [T, 29, 10]')
print(f'날짜 범위           : {dates[0]}  ~  {dates[-1]}')

In [ ]:
train_ds, val_ds, test_ds, scaler_out, scaler_target = prepare_and_split_data(
    raw_features,
    outlet_node_idx = OUTLET_IDX,
    lookback_window = LOOKBACK,
    train_ratio     = TRAIN_RATIO,
    val_ratio       = VAL_RATIO,
    apply_log1p     = True
)

print(f'Train / Val / Test 샘플 수: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')

# DataLoader
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# 배치 shape 확인
bx, by, bidx = next(iter(train_loader))
print(f'배치 X shape: {bx.shape}   (기대: [B, L, 29, 11])')
print(f'배치 y shape: {by.shape}   (기대: [B, 1])')


## 3. Graph Edge 정의

In [ ]:
# Upstream → Downstream 단방향 엣지
edges_list = [
    (20, 5), (5, 23), (21, 24), (22, 24), (24, 23), (23, 25), (4, 25), (19, 25),
    (25, 27), (15, 3), (3, 27), (6, 27), (10, 27), (16, 27), (27, 0), (0, 28),
    (26, 2), (2, 1), (12, 1), (1, 28), (7, 28), (8, 28), (9, 28)
]
source = torch.tensor([e[0] for e in edges_list], dtype=torch.long)
target = torch.tensor([e[1] for e in edges_list], dtype=torch.long)
edge_index = torch.stack([source, target], dim=0).to(DEVICE)
print('Edge index shape:', edge_index.shape)
print('방향: Upstream → Downstream (단방향)')

## 3.5 비딥러닝(Tree) 앙상블 베이스라인 평가
시계열/공간 피처를 단순 1D Flatten으로 다루었을 때의 성능 비교

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 트리 앙상블을 위한 플래튼(Flatten) 데이터 변환 함수
def extract_flat_features(dataset):
    X, Y = [], []
    for x_seq, y_target, _ in dataset:
        X.append(x_seq.numpy().flatten())  # 시공간 피처 1D로 쫙 펼치기
        Y.append(y_target.numpy())
    return np.array(X), np.array(Y)

X_train_flat, Y_train_flat = extract_flat_features(train_ds)
X_test_flat, Y_test_flat   = extract_flat_features(test_ds)

lgb_train = lgb.Dataset(X_train_flat, label=Y_train_flat.ravel())
params = {
    'objective': 'regression',
    'metric': 'mse',
    'learning_rate': 0.05,
    'num_leaves': 15,
    'random_state': 42,
    'verbose': -1
}
gbm = lgb.train(params, lgb_train, num_boost_round=100)
preds_lgb_scaled = gbm.predict(X_test_flat)

# 역변환 평가 (expm1 적용)
# 모델에 들어간 target은 스케일링 + log1p 된 상태임
true_lgb = np.expm1(scaler_target.inverse_transform(Y_test_flat.reshape(-1, 1)).ravel())
pred_lgb = np.expm1(scaler_target.inverse_transform(preds_lgb_scaled.reshape(-1, 1)).ravel())

print('=' * 40)
print(f'[LightGBM Baseline] R² Score:   {r2_score(true_lgb, pred_lgb):.4f}')
print(f'[LightGBM Baseline] RMSE(μg/L): {np.sqrt(mean_squared_error(true_lgb, pred_lgb)):.4f}')
print(f'[LightGBM Baseline] MAE (μg/L): {mean_absolute_error(true_lgb, pred_lgb):.4f}')
print('=' * 40)


## 4. 모델 초기화

In [ ]:
# ── 하이퍼파라미터 ───────────────────────────
IN_FEATURES     = FEATURE_DIM   # 11
TEMPORAL_HIDDEN = 64
GCN_HIDDEN      = 32
NUM_EPOCHS      = 200
LR              = 5e-4
PATIENCE        = 30
# ─────────────────────────────────────────────

# Model A: Transformer 기반 하이브리드 GNN
model_trans = SpatioTemporalHybridGNN(
    in_features      = IN_FEATURES,
    temporal_hidden  = TEMPORAL_HIDDEN,
    gcn_hidden       = GCN_HIDDEN,
    temporal_type    = 'transformer'
).to(DEVICE)

# Model B: GRU 기반 하이브리드 GNN
model_gru = SpatioTemporalHybridGNN(
    in_features      = IN_FEATURES,
    temporal_hidden  = TEMPORAL_HIDDEN,
    gcn_hidden       = GCN_HIDDEN,
    temporal_type    = 'gru'
).to(DEVICE)

print(f'Transformer 모델 파라미터: {sum(p.numel() for p in model_trans.parameters() if p.requires_grad):,}')
print(f'GRU 모델 파라미터:         {sum(p.numel() for p in model_gru.parameters() if p.requires_grad):,}')


## 5. 학습 루프

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, edge_index=None):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch, _ in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()
        if edge_index is not None:
            preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
        else:
            preds = model(x_batch, outlet_node_idx=OUTLET_IDX)
        loss = criterion(preds, y_batch.view_as(preds))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, edge_index=None):
    model.eval()
    total_loss = 0.0
    for x_batch, y_batch, _ in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        if edge_index is not None:
            preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
        else:
            preds = model(x_batch, outlet_node_idx=OUTLET_IDX)
        loss = criterion(preds, y_batch.view_as(preds))
        total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)

print('학습 함수 정의 완료')

In [ ]:
import time
def train_and_eval_model(model_name, model, train_loader, val_loader, test_loader, edge_index):
    print(f'\n>>>>> Training {model_name} <<<<<')
    criterion  = nn.MSELoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

    best_val_loss  = float('inf')
    patience_count = 0
    
    start_time = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, edge_index)
        val_loss   = evaluate(model, val_loader, criterion, edge_index)
        scheduler.step(val_loss)

        if epoch % 20 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d} | Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            torch.save(model.state_dict(), f'best_{model_name}.pt')
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f'  [Early Stopping] Epoch {epoch} (Best Val: {best_val_loss:.4f})')
                break
                
    model.load_state_dict(torch.load(f'best_{model_name}.pt', map_location=DEVICE))
    print(f'  [Done] Training Done: {time.time() - start_time:.1f} sec')
    return model

model_trans = train_and_eval_model('Transformer_GNN', model_trans, train_loader, val_loader, test_loader, edge_index)
model_gru   = train_and_eval_model('GRU_GNN', model_gru, train_loader, val_loader, test_loader, edge_index)


## 6. 학습 곡선 시각화

In [ ]:
# 시각화 생략 (모델 2개 비교를 위해 평가 파트로 직행)


## 7. 테스트 평가

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def get_inverse_metrics(dataset_loader, model_list, edge_index):
    # Retrieve raw responses
    all_true = []
    all_preds_trans = []
    all_preds_gru   = []
    
    for m in model_list:
        m.eval()
        
    with torch.no_grad():
        for x_batch, y_batch, _ in dataset_loader:
            x_batch = x_batch.to(DEVICE)
            
            p_trans = model_list[0](x_batch, edge_index, outlet_node_idx=OUTLET_IDX).cpu().numpy()
            p_gru   = model_list[1](x_batch, edge_index, outlet_node_idx=OUTLET_IDX).cpu().numpy()
            
            all_true.append(y_batch.numpy())
            all_preds_trans.append(p_trans)
            all_preds_gru.append(p_gru)
            
    # Concat
    all_true = np.concatenate(all_true, axis=0) # [N, 1]
    all_preds_trans = np.concatenate(all_preds_trans, axis=0)
    all_preds_gru   = np.concatenate(all_preds_gru, axis=0)
    
    # [핵심] 스케일 역변환 + Expm1 (원래 단위인 μg/L로 복원)
    true_raw  = np.expm1(scaler_target.inverse_transform(all_true)).ravel()
    trans_raw = np.expm1(scaler_target.inverse_transform(all_preds_trans)).ravel()
    gru_raw   = np.expm1(scaler_target.inverse_transform(all_preds_gru)).ravel()
    
    return true_raw, trans_raw, gru_raw

def print_metrics(model_name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    print(f'[{model_name}] R²: {r2:7.4f} | RMSE: {rmse:6.2f} | MAE: {mae:6.2f}')

# Train Evaluation
tr_true, tr_trans, tr_gru = get_inverse_metrics(train_loader, [model_trans, model_gru], edge_index)
print("=== [Train Set] 원단위 (μg/L) 성능 지표 ===")
print_metrics("Trans-GNN", tr_true, tr_trans)
print_metrics("GRU-GNN  ", tr_true, tr_gru)
print()

# Test Evaluation
te_true, te_trans, te_gru = get_inverse_metrics(test_loader, [model_trans, model_gru], edge_index)
print("=== [Test Set] 원단위 (μg/L) 성능 지표 ===")
print_metrics("Trans-GNN", te_true, te_trans)
print_metrics("GRU-GNN  ", te_true, te_gru)

# Persistence Baseline (역변환 적용)
def get_persistence(dataset):
    preds, trues = [], []
    for i in range(len(dataset)):
        x_seq, y_target, _ = dataset[i]
        last_chl_scaled = x_seq[-1, -1, 5].item()
        preds.append(last_chl_scaled)
        trues.append(y_target.item())
        
    p_raw = np.expm1(scaler_target.inverse_transform(np.array(preds).reshape(-1,1))).ravel()
    t_raw = np.expm1(scaler_target.inverse_transform(np.array(trues).reshape(-1,1))).ravel()
    return t_raw, p_raw

persist_true, persist_pred = get_persistence(test_ds)
print_metrics("Persist. ", persist_true, persist_pred)


In [ ]:
# 예측값 vs 실제값 시각화 및 R² 평가
import numpy as np
from sklearn.metrics import r2_score

def get_predictions_and_r2(loader, model, scaler, edge_index):
    all_preds, all_true = [], []
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch, _ in loader:
            x_batch = x_batch.to(DEVICE)
            preds = model(x_batch, edge_index, outlet_node_idx=OUTLET_IDX)
            all_preds.append(preds.cpu().numpy())
            all_true.append(y_batch.numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_true  = np.concatenate(all_true,  axis=0)

    # inverse_transform (chl만 복원)
    dummy = np.zeros((len(all_preds), 10), dtype=np.float32)
    dummy[:, 5] = all_preds.ravel()
    preds_orig = scaler.inverse_transform(dummy)[:, 5]

    dummy[:, 5] = all_true.ravel()
    true_orig = scaler.inverse_transform(dummy)[:, 5]

    r2_val = r2_score(true_orig, preds_orig)
    return true_orig, preds_orig, r2_val

# Train 데이터셋 R² 계산
_, _, train_r2 = get_predictions_and_r2(train_loader, model_hybrid, scaler_out, edge_index)

# Test 데이터셋 R² 계산 및 시각화용 데이터 획득
true_orig, preds_orig, test_r2 = get_predictions_and_r2(test_loader, model_hybrid, scaler_out, edge_index)

print(f'========================================')
print(f'Train R² Score : {train_r2:.4f}')
print(f'Test  R² Score : {test_r2:.4f}')
print(f'========================================\n')

plt.figure(figsize=(12, 4))
plt.plot(true_orig,  label='Observed Chl-a (Test)',  linewidth=1.5)
plt.plot(preds_orig, label='Predicted Chl-a (Test)', linewidth=1.5, linestyle='--')
plt.xlabel('Test Timestep')
plt.ylabel('Chl-a (μg/L)')
plt.title('Hybrid GNN — Test Predictions vs Observations')
plt.legend()
plt.tight_layout()
plt.savefig('test_predictions.png', dpi=150)
plt.show()